In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/aurlieduchez/bankcsv/bank.xls
/kaggle/input/notebooks/janiobachmann/bank-marketing-campaign-opening-a-term-deposit/__notebook__.ipynb
/kaggle/input/notebooks/janiobachmann/bank-marketing-campaign-opening-a-term-deposit/__output__.json
/kaggle/input/notebooks/janiobachmann/bank-marketing-campaign-opening-a-term-deposit/__results__.html
/kaggle/input/notebooks/janiobachmann/bank-marketing-campaign-opening-a-term-deposit/custom.css
/kaggle/input/notebooks/janiobachmann/bank-marketing-campaign-opening-a-term-deposit/__results___files/__results___10_1.png
/kaggle/input/notebooks/janiobachmann/bank-marketing-campaign-opening-a-term-deposit/__results___files/__results___11_0.png
/kaggle/input/notebooks/janiobachmann/bank-marketing-campaign-opening-a-term-deposit/__results___files/__results___13_1.png
/kaggle/input/notebooks/janiobachmann/bank-marketing-campaign-opening-a-term-deposit/__results___files/__results___34_1.png
/kaggle/input/notebooks/janiobachmann/bank-marke

In [3]:
import duckdb
duckdb.sql("""
CREATE OR REPLACE VIEW banque AS
SELECT * FROM read_csv_auto('/kaggle/input/datasets/aurlieduchez/bankcsv/*.xls')
""")  



In [4]:
duckdb.sql("SELECT * FROM banque LIMIt 10").df()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,False,2343,True,False,unknown,5,may,1042,1,-1,0,unknown,True
1,56,admin.,married,secondary,False,45,False,False,unknown,5,may,1467,1,-1,0,unknown,True
2,41,technician,married,secondary,False,1270,True,False,unknown,5,may,1389,1,-1,0,unknown,True
3,55,services,married,secondary,False,2476,True,False,unknown,5,may,579,1,-1,0,unknown,True
4,54,admin.,married,tertiary,False,184,False,False,unknown,5,may,673,2,-1,0,unknown,True
5,42,management,single,tertiary,False,0,True,True,unknown,5,may,562,2,-1,0,unknown,True
6,56,management,married,tertiary,False,830,True,True,unknown,6,may,1201,1,-1,0,unknown,True
7,60,retired,divorced,secondary,False,545,True,False,unknown,6,may,1030,1,-1,0,unknown,True
8,37,technician,married,secondary,False,1,True,False,unknown,6,may,608,1,-1,0,unknown,True
9,28,services,single,secondary,False,5090,True,False,unknown,6,may,1297,3,-1,0,unknown,True


Ici je regarde le contenu de mon tableau que j'ai appelé banque (en créant une vue SQL pour éviter de devoir taper à chaque fois le chemin d'accès à ce fichier).

In [5]:
duckdb.sql("SELECT COUNT(*) FROM banque").df()

,count_star()
0,11162


On a bien 11162 lignes dans notre tableau. Chaque ligne représente un client avec ses caractéristiques.

In [6]:
duckdb.sql("DESCRIBE banque").df()

,column_name,column_type,null,key,default,extra
0,age,BIGINT,YES,None,None,None
1,job,VARCHAR,YES,None,None,None
2,marital,VARCHAR,YES,None,None,None
3,education,VARCHAR,YES,None,None,None
4,default,BOOLEAN,YES,None,None,None
5,balance,BIGINT,YES,None,None,None
6,housing,BOOLEAN,YES,None,None,None
7,loan,BOOLEAN,YES,None,None,None
8,contact,VARCHAR,YES,None,None,None
9,day,BIGINT,YES,None,None,None


On obtient la liste des colonnes avec leur type.

In [7]:
#display(duckdb.sql("SELECT COUNT(*) FROM banque WHERE job='unknown'").df())
#display(duckdb.sql("SELECT COUNT(*) FROM banque WHERE education='unknown'").df())
#display(duckdb.sql("SELECT COUNT(*) FROM banque WHERE contact='unknown'").df())
#display(duckdb.sql("SELECT COUNT(*) FROM banque WHERE poutcome='unknown'").df())

duckdb.sql("""
SELECT 'Job' AS colonne, COUNT(*) AS nb_uknown FROM banque WHERE job='unknown'
UNION ALL
SELECT 'Education', COUNT(*) FROM banque WHERE education='unknown'
UNION ALL
SELECT 'Contact', COUNT(*) FROM banque WHERE contact='unknown'
UNION ALL
SELECT 'poutcome', COUNT(*) FROM banque WHERE poutcome='unknown'
""").df()

,colonne,nb_uknown
0,Job,70
1,Education,497
2,Contact,2346
3,poutcome,8326


Dans le bloc de code précédent, j'ai listé de deux manière différentes le nombre de valeurs inconnues ('unknown) qui figuraient dans les colonnes: job, education, contact et poutcome. La deuxième méthode, plus jolie, crée un tableau grâce aux UNION ALL et en ajoutant les titres "colonne" et "nb_unknown".


In [8]:
duckdb.sql("""
SELECT 'Job' AS colonne, COUNT(*) AS nb_NULL FROM banque WHERE job IS NULL
UNION ALL
SELECT 'Education', COUNT(*) FROM banque WHERE education IS NULL
""").df()

,colonne,nb_NULL
0,Job,0
1,Education,0


En utilisant la même méthode que précédemment pour tester si une case du tableau est vide (NULL), on remarque que le tableau banque ne comporte aucune valeur manquante.
En conclusion, les colonnes JOB et EDUCATION n'ont pas de valeur manquante et peu de valeurs unknown (70/11162 et 497/11162). La colonne contact a 2346/11162 soit 21% de valeurs unknown. Par contre la colonne poutcome a 8326/11162 soit 75% de valeurs inconnues, on ne pourra donc pas l'utiliser pour la suite des analyses. 

La colonne deposit contient des true et des Falses, dans le code suivant, je transforme ces true/False en 0 et 1 afin de pouvoir faire des calculs de pourcentage par la suite en utilisant cette variable (un ordinateur ne peut pas faire des opérations mathématiques sur du texte brut).

In [9]:
duckdb.sql("""
SELECT deposit,
CASE 
WHEN deposit ='True' THEN 1
WHEN deposit='False' THEN 0
ELSE 0
END AS Souscrit
FROM banque 
LIMIT 10
""").df()

,deposit,Souscrit
0,True,1
1,True,1
2,True,1
3,True,1
4,True,1
5,True,1
6,True,1
7,True,1
8,True,1
9,True,1


In [10]:
duckdb.sql("""
SELECT 
CASE 
WHEN age < 30 THEN 'Moins de 30 ans'
WHEN age BETWEEN 30 AND 49 THEN '30-49 ans'
ELSE '50 ans et plus'
END AS tranche_age,
COUNT(*) AS nombre
FROM banque 
GROUP BY tranche_age
""").df()

,tranche_age,nombre
0,30-49 ans,6946
1,50 ans et plus,2665
2,Moins de 30 ans,1551


En faisant la somme du nombre de clients par tranche d'age, on se rend compte qu'on a oublié aucun client (ils ont tous été classé dans une tranche d'age particulière).

In [11]:
duckdb.sql("""
SELECT job,
CASE 
WHEN job='unknown'  THEN 'Non renseigné'
ELSE job
END AS job_new,
FROM banque
LIMIT 5
""").df()

,job,job_new
0,admin.,admin.
1,admin.,admin.
2,technician,technician
3,services,services
4,admin.,admin.


Le fait de remplacer les 'unknown' par 'non renseigné' permet d'indiquer dans le tableau que le client ne nous a pas indiqué son métier (au lieu de se retrouver avec une case vide). Comme on l'a vu précédemment, cela ne concerne que 70 clients (sur 11162).

In [12]:
duckdb.sql("""
CREATE OR REPLACE VIEW banque_propre AS
SELECT age,
CASE 
WHEN age < 30 THEN 'Moins de 30 ans'
WHEN age BETWEEN 30 AND 49 THEN '30-49 ans'
ELSE '50 ans et plus'
END AS tranche_age,
CASE 
WHEN job='unknown'  THEN 'Non renseigné'
ELSE job
END AS job_new,
marital, education, balance, housing, loan, month, 
CASE
WHEN deposit = 'True' THEN 1
ELSE 0
END AS souscrit 
FROM banque
""")  

duckdb.sql("""
SELECT COUNT(*) from banque_propre
""")  

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        11162 │
└──────────────┘

On se rend compte qu'on n'a perdu aucune ligne en créant ce nouveau tableau. La colonne duration (durée de l'appel) a été mise de côté car c'esdt une donnée qui est donnée à la fin de l'appel, elle ne peut donc pas servir à prédire à l'avance si le client va accepter ou non le contrat.

In [13]:
duckdb.sql("""
SELECT ROUND(AVG(souscrit)*100,1) AS taux_global
FROM banque_propre
""").df()

,taux_global
0,47.4


On voit que 47,4% des clients ont dit oui à la démarche téléphonique soit pratiquement la moitié des clients appelés.

In [14]:
duckdb.sql("""
SELECT job_new,ROUND(AVG(souscrit)*100,1) AS taux_par_emploi, COUNT(*) AS nb_clients
FROM banque_propre
GROUP BY job_new
ORDER BY taux_par_emploi DESC
""").df()

,job_new,taux_par_emploi,nb_clients
0,student,74.7,360
1,retired,66.3,778
2,unemployed,56.6,357
3,management,50.7,2566
4,Non renseigné,48.6,70
5,admin.,47.3,1334
6,self-employed,46.2,405
7,technician,46.1,1823
8,services,40.0,923
9,housemaid,39.8,274


Les 3 métiers disant le plus souvent lui sont les étudiants à 74,7%, les retraités à 66,3% et les personnes sans emploi à 56.6%.
Les 3 métiers qui disent le moins souvent oui à cette offre sont les femmes de ménage (39.8%), les entrepreneur (37.5%) et les ouvriers (36.4%).
Le nombre de clients est aussi important à mettre en parallèle à ces conclusions car 36.4% de 1944 ouvriers représente 708 clients alors que 74.7% de 360 étudiants 269 étudiants.

Dans la partie suivante, nous faisons la même étude avec les tranches d'age afin de répondre à la question: dans quelle tranche d'age se trouvent les clients qui ont le plus tendance à répondre favorablement à cette offre téléphonique.

In [15]:
duckdb.sql("""
SELECT tranche_age,ROUND(AVG(souscrit)*100,1) AS taux_par_tranche_d_age, COUNT(*) AS nb_clients
FROM banque_propre
GROUP BY tranche_age
ORDER BY taux_par_tranche_d_age DESC
""").df()

,tranche_age,taux_par_tranche_d_age,nb_clients
0,Moins de 30 ans,59.8,1551
1,50 ans et plus,52.0,2665
2,30-49 ans,42.8,6946


On voit que 59.8% des moins de 30 ans répondent favorablement à cette offre tandis que 42.8% des 30-49 répondent favorablement. L'age du client n'est donc pas un critère majeur qui nous aidera à déterminer le type de client qui répondra favorablement à cette offre.

In [20]:
duckdb.sql("""
SELECT month,ROUND(AVG(souscrit)*100,1) AS taux_par_mois, COUNT(*) AS nb_clients
FROM banque_propre
GROUP BY month
ORDER BY taux_par_mois DESC
""").df

<bound method PyCapsule.df of ┌─────────┬───────────────┬────────────┐
│  month  │ taux_par_mois │ nb_clients │
│ varchar │    double     │   int64    │
├─────────┼───────────────┼────────────┤
│ dec     │          90.9 │        110 │
│ mar     │          89.9 │        276 │
│ sep     │          84.3 │        319 │
│ oct     │          82.4 │        392 │
│ apr     │          62.5 │        923 │
│ feb     │          56.8 │        776 │
│ aug     │          45.3 │       1519 │
│ jun     │          44.7 │       1222 │
│ nov     │          42.7 │        943 │
│ jul     │          41.4 │       1514 │
│ jan     │          41.3 │        344 │
│ may     │          32.8 │       2824 │
├─────────┴───────────────┴────────────┤
│ 12 rows                    3 columns │
└──────────────────────────────────────┘
>

Pour cette étude par contre, on se rend compte que la réponse des clients varie fortement en fonction du mois de l'année durant lequel est passé l'appel téléphonique avec 90.9% de réponses positives en decembre et 89.8% de réponses positives en mars contre seulenebt 32.8% de réponses positives en mai. Néanmoins ici encore il faut faire attention au nombre de clients interrogés chaque moi. En Décembre, cela correspond donc à 100 clients et 926 clients en mai La banque devrait donc appeler davantage les clients en decembre mars octobre et septembre où le taux de réponses positives est beaucoup plus élevé, au lieu de faire beaucoup d'appels en mai avec un faible taux de réussite.


On se pose maintenant la question si avoir un prêt immobilier a un impact sur la décision du client.

In [17]:
duckdb.sql("""
SELECT housing,ROUND(AVG(souscrit)*100,1) AS taux_et_housing, COUNT(*) AS nb_clients
FROM banque_propre
GROUP BY housing
ORDER BY taux_et_housing DESC
""").df()

,housing,taux_et_housing,nb_clients
0,False,57.0,5881
1,True,36.6,5281


On voit donc que pratiquement autant de clients ayant un prêt immobilier que de clients n'en n'ayant pas ont été interrogé et que 57% des clients qui ont souscrit à l'offre n'avaient pas de prêt immobilier.

In [18]:
duckdb.sql("""
SELECT housing,loan,ROUND(AVG(souscrit)*100,1) AS taux_et_housing_et_loan, COUNT(*) AS nb_clients
FROM banque_propre
GROUP BY housing, loan
ORDER BY taux_et_housing_et_loan DESC
""").df()

,housing,loan,taux_et_housing_et_loan,nb_clients
0,False,False,59.6,5256
1,True,False,37.6,4446
2,False,True,35.0,625
3,True,True,31.7,835


Donc les clients qui répondent le plus positivement sont ceux qui ont souscrit à aucun prêt (59,6%).

En conclusion, l'absence de crédit est le facteur le plus déterminant: les clients sans aucun prêt (housing = False et loan = False) atteignent un taux de souscription de 59,6 % sur 5 256 clients, alors que la présence d'un prêt immobilier fait chuter ce taux à 36,6 % sur 5 281 clients.

Les profils disposant de plus de disponibilité souscrivent nettement plus, menés par les étudiants (74,7 % sur 360 clients) et les retraités (66,3 % sur 778 clients), tandis que les ouvriers affichent le taux le plus faible à 36,4 % malgré un fort volume de 1 944 clients.

(Hypothèse nuancée par le volume) : L'âge influe peu de manière linéaire (les 30-49 ans souscrivent le moins avec 42,8 % sur 6 946 clients), et si certains mois comme décembre atteignent 90,9 % de succès, ce résultat doit être nuancé car il ne concerne que 110 clients, contre seulement 32,8 % de réussite en mai sur un volume massif de 2 824 clients.

In [22]:
duckdb.sql("""
COPY banque_propre TO '/kaggle/working/banque_propre.csv' (HEADER, DELIMITER ',');
""")

L'objectif du code précédent est d'exporter la vue propre en fichier CSV pour Tableau.
Vérification avec Python du format du fichier csv ainsi crée:

In [24]:
import pandas as pd
# Lecture du fichier exporté
df_verif=pd.read_csv('/kaggle/working/banque_propre.csv')

# Affiche le nombre de lignes et de colonnes
print("Dimensions du fichier:",df_verif.shape)

# Affichage des 5 premieres lignes
df_verif.head()

Dimensions du fichier: (11162, 10)


,age,tranche_age,job_new,marital,education,balance,housing,loan,month,souscrit
0,59,50 ans et plus,admin.,married,secondary,2343,True,False,may,1
1,56,50 ans et plus,admin.,married,secondary,45,False,False,may,1
2,41,30-49 ans,technician,married,secondary,1270,True,False,may,1
3,55,50 ans et plus,services,married,secondary,2476,True,False,may,1
4,54,50 ans et plus,admin.,married,tertiary,184,False,False,may,1
